
# Nearest Prototype Classifier Model
----

### Extracting Features 
The model infers the language a text is written in by analysing trigrams within the text. In future this model will be updated in future to analyse n-grams within the text, and analysis will be performed to find the value of n which produces the most accurate model.

The model works by performing the following steps:

1. (Training) The model calculates the frequency of trigrams in each training language
2. (Classifying) A text is provided to classify the model calculates the frequencies of its trigrams
3. The model will loop through each language and its trigrams

    a. If a trigram is shared between the incoming text and the current trained language, the model calculates the euclidean inner product between the matching trigram frequencies

    b. This value is then added to the score for the current language
4. The language with the largest score is inferred as the language the incoming text is written in

---------


We first import all modules we will need for the model.

In [64]:
from collections import defaultdict as dd
import csv
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt 
import numpy as np
import pickle

# We expect to deal with large csv files when training the model, so adjust the csv field size limit
csv.field_size_limit(100000000)

100000000

# Cleaning training data
We first clean the training data (collected form various databases accross the internet) so all the csv files are in a consistent format.
We want the csv files to have the following format: language, text1, text2, ...

All of the files in ./data_to_clean are in the form text, language so we will just swap these two columns. Note train.csv is already in the desired format.

In [65]:
count = 1
for filename in ['12langs','EnGerFrenSpan','languages']:
        with open(f'./data_to_clean/{filename}.csv', 'r', encoding='utf8') as infile, \
        open(f'./training_data/training_file_{count}_cleaned.csv', 'w', encoding='utf8', newline='') as outfile:

                writer = csv.DictWriter(outfile, fieldnames=['language', 'text'])

                for row in csv.DictReader(infile):
                        writer.writerow(row)

        count += 1

# Building the model

## Calculating trigrams in the training data
Below is a function to find the trigrams and their counts (number of occurences) in a given text.

In [66]:

def count_trigrams(document):
    '''
    count_trigrams takes a string and returns a dictionary of the counts 
    of trigrams within the document (the language vector)
    the input 'document' is a string
    '''

    language_vector = dd(int)

    if len(document) < 3:
        return language_vector
    
    for i in range(len(document) - 2):
        language_vector[document[i] + document[i+1] + document[i+2]] += 1

    return language_vector


# todo in future: adapt this code to calculate n-grams so we can improve the accuracy of the model by optimising n.


## Training the classifier
We now train the classifier.

The maths used in building the model is as follows:
1. We create an inner product space V (vector space with euclidean inner product) with the basis vectors being all possible trigrams that can be made with any languages characters
2. For each text, a vector is created in V depending on the counts of each traigram
3. Once all texts in a language are trained, a vector representing each languages own trigrams and counts is created, and then normalised - call these vectors the 'language vectors'
4. When a text is given to the model to classify, we create a vector in the vector space determined by its trigrams and thier counts. We then calculate the euclidean inner product between this vector and all language vectors. The largest inner product represents the language the text is inferred to be written in.

Note: Because we normalised all the language vectors, the inner product we performed is finding the magnitude of the projection of the incoming vector onto each of the language vectors. This allows us to easily infer which language the incoming text is written in.


## Normalise Vectors
Below is a function to normalise the trigram counts for a given text (turning the counts into frequencies).

In [67]:
def normalise(language_vector):
    '''
    normalise the values in a dictionary of trigram counts, returning them in a dictionary
    '''
    #calculate the magnitude of the dictionaries values (each value is taken to be a scaled basis vector, with each basis vector being a unique trigram)
    # each trigram is taken to be a basis vector in the vector space of a language's trigrams.
    # so to normalise the trigrams we scale all vectors in trigram_counts 
    magnitude = sqrt(sum(x**2 for x in language_vector.values()))

    for key in language_vector:
        language_vector[key] /= magnitude

    return language_vector



## Creating language vectors

We now write a function to read texts from csv files in a training directory. Recall the csv files all have format language, text1, text2. We will return a dictionary of languages with their normalised trigram counts (or normalised language vectors) in the form *{language1: trigram_counts1, Language2: trigram_counts2, ...}*.

In future we should try and clean data to better train the model on - e.g. remove trigrams found in links, code, etc.

In [68]:

def train_classifier(training_directory):
    '''
    takes list of file names to train the model on and returns a dictionary of normalised language vectors.
    '''
    language_vectors = dd()     # the dictionary with all language vectors


    for filename in training_directory:
        with open(filename, 'r', encoding='utf8') as fp:
            for line in csv.reader(fp):
                line = list(line)

                # use .upper() so we don't get duplicate languages 
                language = line[0].upper()

                # add trigrams counts to this vector
                create_language_vector = dd(int)

                for text in line[1:]:
                    create_language_vector.update(count_trigrams(text))

                # check if we have seen this language before (i.e. whether the language vector exists or not)
                if language in language_vectors:
                    # if the language vector exists add it to its existing language vector
                    for trigram in create_language_vector:
                        language_vectors[language][trigram] += create_language_vector[trigram]
                else:
                    language_vectors[language] = create_language_vector
    

    # don't normalise vectors until now in case there is more texts in an alreay seen language later in the training data
    for vector in language_vectors:
        language_vectors[vector] = normalise(language_vectors[vector])
    

    return language_vectors

## Scoring Documents
We now move on to scoring input documents (performing the above mentioned euclidean inner product).

In future, the below algorithm should be made more efficient, as it is currently somewhat slow. For example, representing data in matrices rather than dictionaries could help speed up matrix multiplication in the inner product calculation. Something similar should also be looked at to try and avoid looping through trained_trigrams, as what happens at the bottom of the above code block.

In [69]:
def score_document(document_to_classify, language_vectors):
    '''
    takes in a document to classify (string) and the dictionary of language vectors and returns the scores for each language in language vectors
    '''


    document_to_classify_trigrams = count_trigrams(document_to_classify)
    languages_scores = dd(int)
        
    for vector in language_vectors:
        current_language_vector = language_vectors[vector]
            
        for trigram in document_to_classify_trigrams:
            if trigram in current_language_vector:
                new_score = document_to_classify_trigrams[trigram] * current_language_vector[trigram]
                languages_scores[vector] += new_score
    
    
    return languages_scores

## Classifying Documents
We now implement a method to classify the language a document has been written in.

We do this by finding the language with the highest score in languages_scores and returning it.

To do this we sort the keys in score_document according to the magnitude of their values. We use a tolerance of 1e-10. Should a tie exist in this range, we return both possible languages and ask the user to enter more text.

In future we should allow the user to list out the k most likely languages.

In [79]:
def classify_doc(document_to_classify, language_vectors, tolerance, classifying_file=False):
    '''
    The document to classify can be either a string (classifying_file=False by default) or a text file (classifying_file=true)
    language_vectors is the dictionary of language vectors
    
    tolerance the min difference in scores two languages can have and be given different ranks
    returns the language a text is written in, or, if a tie exists, a list of possible languages and a prompt to enter more text
    '''    

    if classifying_file:
        document_to_classify = open(str(document_to_classify)).read()

    # returns dictionary of all languages scores
    document_to_classify_scores = score_document(document_to_classify, language_vectors)
    
    # initialise the most_common_language to be any of the languages
    most_common_language = list(document_to_classify_scores.keys())[0]
    
    tie = [most_common_language]

    
    for language in document_to_classify_scores:
        if (document_to_classify_scores[most_common_language] - document_to_classify_scores[language]) > 1e-10:
            continue
        elif (document_to_classify_scores[most_common_language] - document_to_classify_scores[language]) < -(1e-10):
            most_common_language = language
            tie = [most_common_language]
        else:
            tie.append(language)

    # if no tie
    if len(set(tie)) == 1:
        return most_common_language

    else:
        print('There is a tie. The possible languages are as follows:')
        print(tie)
        print('For a more accurate model, please input more text')



## Training the model
To train the model, we run the following code. We export the trained data to avoid needing to re-train the model everytime we want to classify a document. We export the data to a pickle file as they are fast and allow us to store dictionaries (language vectors).

In [71]:
training_files = [f'./training_data/training_file_{i}_cleaned.csv' for i in range(4)]

trained_data = train_classifier(training_files)
with open('trained_data.pkl', 'wb') as fpx:
    pickle.dump(trained_data, fpx)



To classify a document or text, run the following code

In [72]:
with open('traineddata.pkl', 'rb') as newfpx:
    trained_data = pickle.load(newfpx)

In [83]:
print(classify_doc('this is some english', trained_data, 1e-10, classifying_file=False))

print('Some other data about the model:')
print(f'Number of languages trained on: {len(trained_data.keys())}')
print(f'Languages trained on: {[key for key in trained_data.keys()]}')


ENGLISH
Some other data about the model:
Number of languages trained on: 85
Languages trained on: ['SWEDISH', 'ICELANDIC', 'ESTONIAN', 'TELUGU', 'PIEMONTESE', 'MARATHI', 'ITALIAN', 'JAVANESE', 'SLOVENIAN', 'NEAPOLITAN', 'LUXEMBOURGISH; LETZEBURGESCH', 'GREEK, MODERN (1453-)', 'CATALAN; VALENCIAN', 'HINDI', 'AZERBAIJANI', 'KOREAN', 'DANISH', 'BULGARIAN', 'LATIN', 'VIETNAMESE', 'HUNGARIAN', 'MACEDONIAN', 'WELSH', 'BOSNIAN', 'GEORGIAN', 'LITHUANIAN', 'MALAY', 'FRENCH', 'NORWEGIAN', 'TURKISH', 'BENGALI', 'LOW GERMAN', 'SUNDANESE', 'RUSSIAN', 'ARAGONESE', 'AFRIKAANS', 'TAMIL', 'CEBUANO', 'BISHNUPRIYA MANIPURI', 'KURDISH', 'FINNISH', 'SERBO-CROATIAN', 'ALBANIAN', 'HEBREW', 'INDONESIAN', 'NEPAL BHASA', 'IDO', 'ASTURIAN', 'NORWEGIAN NYNORSK; NYNORSK, NORWEGIAN', 'LATVIAN', 'ENGLISH', 'HAITIAN; HAITIAN CREOLE', 'THAI', 'CROATIAN', 'PORTUGUESE', 'WALLOON', 'SICILIAN', 'CHINESE', 'GERMAN', 'OCCITAN (POST 1500); PROVENÇAL', 'UKRAINIAN', 'JAPANESE', 'BELARUSIAN', 'GALICIAN', 'CZECH', 'PERSIAN', 'SL

### Plotting language scores
We now visualise the models success in classifying a document.

We first write a function to extract scores for certain languages into a list that can be plotted.

// can probably move this and the following markdowns and functions below up above the scoring section.

------


In [74]:
# code to extract into lists

We now write a function to plot the scores of each language

In [75]:
# plot language scores

### Measruing the performance of the model
We now move onto measuring the performance of our model on a test set by analysing its *precision* and *recall*.\
Precision = number of correct classifications for a given language / total number of classifications made for that language. \
Recall = number of correct classifications for a given language / total number of documents written in that language in the test set.

We want to achieve a good mix of precision and recall in our model - depending largely on the goals of our model.

A model with high precision but low recall is very trustworthy as it is precise in the classifications it makes, but conservative in making classifications.

Similarly, a model with low precision and high recall is good at making lots of classifications but poor when it comes to the accuracy of such classifications.

In [76]:
# precision and recall - should be cleaned, tested and optimised in future

def calc_precision(test_set, language_vectors=trained_data):
    """calc_precision takes the filename of a csv file test_set and returns 
    a dictionary of the precision of the classifier per language."""
    
    
    pre_languages_precision = dd(lambda: [0,0])
    
    
    languages_precision = {}
    for glob_lang in language_vectors:
        languages_precision[glob_lang] = 0
    
    with open(test_set, 'r') as fp:
        for line in csv.reader(fp):
            N_predicted = 0
            N_correct = 0
            
            texts = iter(line)
            actual_language = next(texts)
            
            for text in texts:
                N_predicted += 1          
                N_correct += int(actual_language == classify_doc(text, default_language_vectors))
            
            pre_languages_precision[actual_language][1] += N_predicted
            pre_languages_precision[actual_language][0] += N_correct      

    
    for language in pre_languages_precision:
        data = pre_languages_precision[language]
        if data[1] != 0:            
            languages_precision[language] = data[0] / data[1]
        else:
            languages_precision[language] = 0
    return languages_precision





In [77]:

def calc_recall(test_set, language_vectors=trained_data):
    """calc_recall takes the filename of a csv file test_set and returns
    a dictionary of the recall of the classifier for each language."""
    
    
    languages_recall_data = dd(int)
    languages_recall_results = {}
    
    for glob_lang in language_vectors:
        languages_recall_results[glob_lang] = 0
        
    N_predicted = 0
    with open(test_set, 'r') as fp:
        for line in csv.reader(fp):
            N_correct = 0
            
            texts = iter(line)
            actual_language = next(texts)
            
            for text in texts:
                N_predicted += 1          
                N_correct += int(actual_language == classify_doc(text, default_language_vectors))
            
            languages_recall_data[actual_language] += N_correct      

    
    for language in languages_recall_data:
        data = languages_recall_data[language]
        if N_predicted == 0:            
            languages_recall_results[language] = 0
        else:
            languages_recall_results[language] = data / N_predicted
    return languages_recall_results

